# Enterprise Knowledge Silos — Advanced RAG

Enterprise Knowledge Assistant using:
- LangChain
- LangChain Community
- LangChain Groq
- LangChain HuggingFace
- Docling (layout-aware parsing)
- FAISS Vector Database
- BM25 Keyword Retrieval
- Reciprocal Rank Fusion
- Cross-Encoder Re-ranking
- HuggingFace Embeddings
- Groq Llama 3.3 70B

---

## Pipeline Comparison

### Naive RAG (Previous)

```
Documents
↓
Loader (PyPDFLoader / Markdown)
↓
RecursiveCharacterTextSplitter
↓
HuggingFace Embeddings
↓
FAISS Vector Store
↓
Vector Similarity Search
↓
Prompt Template
↓
LLM Response
```

### Advanced RAG (Current)

```
Documents
↓
Layout-aware Parser (Docling)
↓
Layout-aware Chunking (structure-preserving)
↓
HuggingFace Embeddings
↓
FAISS Vector Store + BM25 Index
↓
User Query
↓
LLM Query Rewriter
↓
Rewritten Query
↓
┌────────────────────────────┐
│  Vector Search    BM25     │
└────────────┬───────────────┘
             ↓
 Reciprocal Rank Fusion
             ↓
 Top 10 Documents
             ↓
 Cross-Encoder Re-ranker
             ↓
 Top 4 Documents
             ↓
 Prompt Template
             ↓
 LLM Response
```


## 1. Install Dependencies

We install all required libraries for the Advanced RAG pipeline:

- **docling** — layout-aware document parsing that preserves headings, tables, captions, and code blocks
- **langchain / langchain-community / langchain-huggingface** — core LangChain ecosystem for chains, retrievers, and embeddings
- **faiss-cpu** — dense vector similarity search
- **rank_bm25** — BM25 sparse keyword retrieval
- **sentence-transformers** — both the embedding model and the Cross-Encoder re-ranker
- **gradio** — interactive front-end UI


In [ ]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-groq \
    langchain-huggingface \
    langchain-text-splitters \
    faiss-cpu \
    rank_bm25 \
    sentence-transformers \
    docling \
    pypdf \
    beautifulsoup4 \
    requests \
    gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.7/604.7 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.7/260.7 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━

## 2. Environment Configuration

Set the Groq API key used to access the Llama 3.3 70B model for both query rewriting and response generation.


In [ ]:
import os
from getpass import getpass

# Prompt for the Groq API key (input is hidden, not echoed to the cell output)
groq_api_key = getpass("Enter your GROQ_API_KEY: ")

# Persist it to a .env file so it can be reloaded in future sessions
with open(".env", "w") as f:
    f.write(f"GROQ_API_KEY={groq_api_key}\n")

# Set it in the current environment for this session
os.environ["GROQ_API_KEY"] = groq_api_key

print("API key saved to .env and loaded into environment.")

Enter your GROQ_API_KEY: ··········
API key saved to .env and loaded into environment.


## 3. Imports

All library imports are consolidated here for clarity. We import:

- **Docling** classes for layout-aware document conversion
- **LangChain** core, community, and HuggingFace components
- **rank_bm25** for keyword-based retrieval
- **SentenceTransformers CrossEncoder** for re-ranking


In [ ]:
import os
import json
import requests
from pathlib import Path
from collections import Counter

from bs4 import BeautifulSoup

# LangChain core
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate

# LangChain community
from langchain_community.vectorstores import FAISS

# LangChain HuggingFace (modern package)
from langchain_huggingface import HuggingFaceEmbeddings

# LangChain Groq
from langchain_groq import ChatGroq

# Docling — layout-aware document parsing
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import PdfFormatOption

# BM25 keyword retrieval
from rank_bm25 import BM25Okapi

# Cross-Encoder re-ranking
from sentence_transformers import CrossEncoder

print("All imports successful.")

/tmp/ipykernel_2271/1305786477.py:14: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


All imports successful.


## 4. Layout-Aware Document Ingestion with Docling

### Why Layout-Aware Parsing?

Standard PDF text extraction (e.g. `PyPDFLoader`) treats a document as a flat stream of characters. It:
- Loses all structural hierarchy (headings, sections, subsections)
- Merges table cells into unstructured text
- Drops figure captions or conflates them with body text
- Cannot distinguish code blocks from prose

For **technical enterprise documentation** — which is typically rich in tables, code examples, and multi-level headings — this loss of structure directly harms retrieval quality. A chunk extracted from a table cell with no heading context is nearly impossible to match to a natural-language query.

**Docling** is a layout-aware parser developed by IBM Research. It uses a document understanding model to:
- Identify and tag document elements: headings, paragraphs, tables, figures, captions, lists, code blocks
- Preserve the logical reading order even in multi-column layouts
- Export to structured Markdown that retains hierarchy

This means every chunk produced downstream will carry meaningful structural context, making retrieval far more reliable.


In [ ]:
def load_with_docling(path: str, doc_type: str) -> list[Document]:
    """
    Load a PDF using Docling's layout-aware document converter.

    Note: OCR is explicitly disabled (do_ocr=False) because:
      1. The knowledge base consists of text-based PDFs — OCR is not needed.
      2. The installed RapidOCR version does not support the PP-OCRv6 model
         that newer Docling versions request by default, causing a ValueError.
    """

    # do_ocr=False  — skips RapidOCR entirely (not needed for text-based PDFs)
    # do_table_structure=True — still extracts and preserves table layout
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = False
    pipeline_options.do_table_structure = True

    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options
            )
        }
    )

    result = converter.convert(path)
    doc = result.document
    markdown_text = doc.export_to_markdown()

    return [Document(
        page_content=markdown_text,
        metadata={
            "source": path,
            "type": doc_type,
            "parser": "docling"
        }
    )]


def load_markdown(path: str) -> list[Document]:
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    return [Document(
        page_content=text,
        metadata={"source": path, "type": "README", "parser": "direct"}
    )]


def load_webpage(url: str, doc_type: str) -> list[Document]:
    html = requests.get(url).text
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(separator=" ", strip=True)

    return [Document(
        page_content=text,
        metadata={"source": url, "type": doc_type, "parser": "beautifulsoup"}
    )]


print("Document loader functions defined.")

Document loader functions defined.


In [ ]:
# ---------------------------------------------------------------
# Load all documents from the knowledge base
# PDFs are loaded with Docling for layout-aware structured output
# Markdown files are loaded directly since they are already structured
# ---------------------------------------------------------------

documents = []

# Load the README (already structured Markdown — no conversion needed)
documents.extend(load_markdown("README.md"))

# Load PDFs using Docling (layout-aware parsing)
documents.extend(load_with_docling("rag_paper.pdf",       doc_type="WHITEPAPER"))
documents.extend(load_with_docling("documentation.pdf",   doc_type="DOCUMENTATION"))
documents.extend(load_with_docling("API.pdf",             doc_type="API_DOC"))
documents.extend(load_with_docling("LangGraph_overview.pdf",             doc_type="DOCUMENTATION"))
documents.extend(load_with_docling("Quickstart_LangChain.pdf",             doc_type="DOCUMENTATION"))
documents.extend(load_with_docling("Quickstart_langgraph.pdf",             doc_type="DOCUMENTATION"))
documents.extend(load_with_docling("Workflows_langgraph.pdf",             doc_type="DOCUMENTATION"))

print(f"Documents Loaded: {len(documents)}")
print(Counter(doc.metadata["type"] for doc in documents))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Documents Loaded: 8
Counter({'DOCUMENTATION': 5, 'README': 1, 'WHITEPAPER': 1, 'API_DOC': 1})


## 5. Layout-Aware Chunking

### Why Layout-Aware Chunking?

**RecursiveCharacterTextSplitter** splits text purely by character count. It has no knowledge of document structure:
- A split can occur in the middle of a table row, destroying the table
- A heading may be placed in a different chunk from the content it describes
- A code block may be split across chunks, making it syntactically incomplete

For **technical enterprise documentation**, this is a significant problem. A chunk containing a table row without its header row, or a paragraph without its section heading, carries very little retrieval signal.

### Our Approach

Since Docling exports documents as structured Markdown, we can split at Markdown structural boundaries:

1. **Primary split** — on Markdown headings (`#`, `##`, `###`). Each section stays together.
2. **Secondary split** — if a section is too large, we split on double newlines (paragraph boundaries), never mid-sentence.
3. **Tables, code blocks** — preserved intact because Docling's Markdown uses standard fenced blocks.

We use `MarkdownHeaderTextSplitter` from LangChain combined with `RecursiveCharacterTextSplitter` as a fallback for oversized sections. This gives us the best of both worlds: structural awareness with a size safety net.

### Why This Improves Retrieval

- Each chunk corresponds to a semantically cohesive unit (a section, a table, a procedure)
- Headings are propagated as metadata to every child chunk, giving context even when a chunk is retrieved in isolation
- Embedding quality improves because the chunk content is coherent rather than mid-sentence fragments


In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

def layout_aware_chunk(documents: list[Document],
                        max_chunk_size: int = 1000,
                        chunk_overlap: int = 150) -> list[Document]:
    """
    Layout-aware chunking strategy for Markdown-structured documents.

    Step 1: Split on Markdown headings to preserve section boundaries.
            Heading text is stored as metadata so context is not lost.

    Step 2: Apply RecursiveCharacterTextSplitter as a secondary pass
            only on sections that exceed max_chunk_size. This prevents
            very large sections from producing oversized embeddings
            while still respecting paragraph boundaries.

    Args:
        documents:       List of LangChain Documents (Markdown content).
        max_chunk_size:  Maximum characters per chunk.
        chunk_overlap:   Overlap for the secondary character splitter.

    Returns:
        List of LangChain Documents — one per chunk.
    """

    # Define heading levels to split on
    # Heading text is stored in metadata for retrieval context
    headers_to_split_on = [
        ("#",   "heading_1"),
        ("##",  "heading_2"),
        ("###", "heading_3"),
    ]

    # Primary splitter — structural (heading-level)
    header_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on,
        strip_headers=False  # keep headings inside chunk content
    )

    # Secondary splitter — character-level fallback for large sections
    # We use paragraph-level separators to avoid mid-sentence splits
    char_splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]  # paragraph-first order
    )

    all_chunks = []

    for doc in documents:
        # Step 1: Split by Markdown headings
        header_chunks = header_splitter.split_text(doc.page_content)

        for hchunk in header_chunks:
            # Merge parent document metadata with heading metadata
            merged_metadata = {**doc.metadata, **hchunk.metadata}

            if len(hchunk.page_content) <= max_chunk_size:
                # Chunk fits within size limit — keep as-is
                all_chunks.append(
                    Document(
                        page_content=hchunk.page_content,
                        metadata=merged_metadata
                    )
                )
            else:
                # Step 2: Section is too large — apply character splitter
                sub_chunks = char_splitter.split_text(hchunk.page_content)
                for sc in sub_chunks:
                    all_chunks.append(
                        Document(
                            page_content=sc,
                            metadata=merged_metadata
                        )
                    )

    return all_chunks


# Apply layout-aware chunking to all loaded documents
chunks = layout_aware_chunk(documents)

print(f"Total Chunks: {len(chunks)}")
print(f"\nSample chunk metadata: {chunks[0].metadata}")
print(f"\nSample chunk content (first 300 chars):\n{chunks[0].page_content[:300]}")

Total Chunks: 198

Sample chunk metadata: {'source': 'README.md', 'type': 'README', 'parser': 'direct'}

Sample chunk content (first 300 chars):
<div align="center">
<a href="https://docs.langchain.com/oss/python/langchain/overview">
<picture>
<source media="(prefers-color-scheme: dark)" srcset=".github/images/logo-dark.svg">
<source media="(prefers-color-scheme: light)" srcset=".github/images/logo-light.svg">
<img alt="LangChain Logo" src="


## 6. Embeddings

We retain the **HuggingFace `sentence-transformers/all-MiniLM-L6-v2`** embedding model from the original pipeline.

This model:
- Produces 384-dimensional dense vectors
- Runs locally — no external API calls required for embedding
- Is fast and well-suited to retrieval tasks

We now import it from the modern `langchain-huggingface` package rather than the deprecated `langchain-community` path.


In [ ]:
# Load the HuggingFace embedding model
# Using langchain-huggingface (modern package, replaces langchain-community embeddings)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings Model Loaded: sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings Model Loaded: sentence-transformers/all-MiniLM-L6-v2


## 7. FAISS Vector Store

We retain **FAISS** as the vector database. FAISS is a high-performance library from Meta for dense vector similarity search.

All document chunks are embedded and stored in FAISS. During retrieval, a query vector is compared against all stored chunk vectors using cosine similarity, and the top-k most similar chunks are returned.

We also persist the index to disk so it can be reloaded without re-embedding.


In [ ]:
# Build FAISS vector store from all document chunks
# This embeds every chunk and indexes the vectors for similarity search
vectorstore = FAISS.from_documents(chunks, embeddings)

print("Vector DB Created")

# Persist the FAISS index to disk for later reuse
vectorstore.save_local("enterprise_vector_db")

print("Vector DB Saved to: enterprise_vector_db")

Vector DB Created
Vector DB Saved to: enterprise_vector_db


## 8. BM25 Keyword Index

### Why BM25 in Addition to Vector Search?

Vector similarity search is excellent at capturing **semantic meaning** — it can match a query like *"how do I persist state between steps"* to a chunk that discusses *"stateful graph execution"*, even without exact word overlap.

However, vector search has a well-known weakness: **exact term matching**. If a user queries a specific function name, class name, or technical acronym (e.g., `RunnablePassthrough`, `MemorySaver`, `LCEL`), a vector search may rank chunks lower simply because the embedding space does not distinguish fine-grained technical tokens well.

**BM25** (Best Match 25) is a classic TF-IDF-based keyword retrieval algorithm. It:
- Rewards chunks that contain the exact query terms
- Applies length normalisation so shorter chunks are not unfairly penalised
- Is extremely fast — no neural inference required

**For enterprise documentation**, users frequently search by exact names: API endpoints, configuration keys, class names, method names. BM25 captures these precisely where vector search may miss.

Combining both retrieval signals via **Reciprocal Rank Fusion** gives us the best of both worlds.


In [ ]:
def build_bm25_index(chunks: list[Document]):
    """
    Build a BM25 index from the document chunks.

    BM25 operates on tokenised text.
    We use simple whitespace tokenisation here.
    For production use, a proper tokeniser (e.g. NLTK or spaCy)
    would improve precision by handling stop words and stemming.

    Returns:
        bm25: The BM25Okapi index object
        tokenised_corpus: List of tokenised chunk texts (for reference)
    """

    # Tokenise each chunk's text by splitting on whitespace and lowercasing
    tokenised_corpus = [
        chunk.page_content.lower().split()
        for chunk in chunks
    ]

    # Build the BM25 index — this computes IDF scores across the corpus
    bm25 = BM25Okapi(tokenised_corpus)

    return bm25, tokenised_corpus


def bm25_retrieve(query: str,
                  bm25: BM25Okapi,
                  chunks: list[Document],
                  top_k: int = 10) -> list[Document]:
    """
    Retrieve the top-k chunks from the BM25 index for a given query.

    Args:
        query:   The search query string.
        bm25:    The BM25Okapi index.
        chunks:  The original document chunks.
        top_k:   Number of top results to return.

    Returns:
        List of top-k LangChain Documents ranked by BM25 score.
    """

    # Tokenise the query the same way the corpus was tokenised
    tokenised_query = query.lower().split()

    # Get BM25 scores for all chunks
    scores = bm25.get_scores(tokenised_query)

    # Sort chunk indices by descending score and take top-k
    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:top_k]

    return [chunks[i] for i in top_indices]


# Build the BM25 index from all document chunks
bm25_index, tokenised_corpus = build_bm25_index(chunks)

print(f"BM25 Index Built. Corpus size: {len(tokenised_corpus)} documents")

BM25 Index Built. Corpus size: 198 documents


## 9. LLM Initialisation

We initialise the Groq Llama 3.3 70B model. This LLM serves two roles in the Advanced RAG pipeline:

1. **Query Rewriter** — rewrites the user's raw query into a precise retrieval query
2. **Response Generator** — generates the final answer from the reranked context

`temperature=0` ensures deterministic, factual outputs for both roles.


In [ ]:
# Initialise the Groq LLM
# temperature=0 for deterministic, factual outputs
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("Groq Llama 3.3 70B Loaded")

Groq Llama 3.3 70B Loaded


## 10. Query Rewriting

### What is Query Rewriting?

Query Rewriting is a pre-retrieval step that uses an LLM to reformulate the user's original query into a form that is better suited for document retrieval.

Users often write queries that are:
- Too vague: *"How does it work?"* — refers to nothing specific
- Conversational: *"What about the agent thing?"* — contains pronouns and implicit context
- Poorly worded: *"langchain chain lcel create"* — keyword soup

The LLM rewrites these into a precise, self-contained retrieval query:

| Original | Rewritten |
|---|---|
| *"How does it work?"* | *"How does LangGraph execute stateful workflows?"* |
| *"What about the agent thing?"* | *"How does LangChain implement autonomous agents?"* |
| *"langchain chain lcel create"* | *"How do you create a chain using LangChain Expression Language (LCEL)?"* |

### Why It Improves Retrieval

Both vector similarity search and BM25 rely on the query text to match relevant documents. A vague or poorly-phrased query produces a poor embedding vector and poor keyword overlap. By rewriting it first, we ensure the query accurately reflects the user's intent in retrieval-friendly language.

### When It Is Most Useful

- Conversational systems where users reference prior turns (*"Tell me more about that"*)
- Domain-specific knowledge bases where users may not know correct terminology
- Long-tail queries where the user's phrasing is highly informal


In [ ]:
# Query Rewriting prompt — instructs the LLM to produce a precise retrieval query
QUERY_REWRITE_PROMPT = PromptTemplate(
    input_variables=["query"],
    template="""You are a query rewriting assistant for an Enterprise Knowledge Base about LangChain documentation.

Your task is to rewrite the user's query into a precise, self-contained retrieval query.

Rules:
- Remove ambiguity and vague pronouns
- Use precise technical terminology relevant to LangChain
- Make the query specific and retrievable
- Return ONLY the rewritten query — no explanation, no preamble

Original Query: {query}

Rewritten Query:"""
)

# Build the query rewriting chain
query_rewrite_chain = QUERY_REWRITE_PROMPT | llm


def rewrite_query(query: str) -> str:
    """
    Rewrite the user's query using the LLM to improve retrieval quality.

    Args:
        query: The original user query.

    Returns:
        A rewritten query string that is more precise and retrieval-friendly.
    """

    result = query_rewrite_chain.invoke({"query": query})

    # Extract the text content from the LLM response
    rewritten = result.content.strip()

    return rewritten


# Demonstrate query rewriting
test_queries = [
    "How does it work?",
    "What about the agent thing?",
    "How do retrievers work in LangChain?"
]

print("Query Rewriting Demonstration:\n")
for q in test_queries:
    rewritten = rewrite_query(q)
    print(f"Original:  {q}")
    print(f"Rewritten: {rewritten}")
    print()

Query Rewriting Demonstration:

Original:  How does it work?
Rewritten: What is the architecture and functionality of LangChain, including its components and interaction mechanisms?

Original:  What about the agent thing?
Rewritten: What is the purpose and functionality of the LangChain Agent in the context of building applications with large language models?

Original:  How do retrievers work in LangChain?
Rewritten: What is the architecture and functionality of retriever components in LangChain, including their role in indexing and querying knowledge bases?



## 11. Hybrid Retrieval with Reciprocal Rank Fusion

### Semantic Search vs Keyword Search

**Semantic Search** (FAISS + dense vectors)
- Understands meaning and intent
- Can match synonyms and paraphrases
- Struggles with rare or exact technical terms

**Keyword Search** (BM25)
- Matches exact terms precisely
- Handles technical names, API endpoints, function signatures
- Cannot understand meaning or handle paraphrase

### Why Enterprise Documentation Needs Both

LangChain documentation contains both conceptual prose (*"Chains allow you to sequence operations..."*) and highly precise technical content (*"`RunnablePassthrough.assign()` adds keys to the pipeline"*). Neither retrieval method alone covers both well.

### Reciprocal Rank Fusion (RRF)

RRF is a score-free rank aggregation method. Rather than combining raw similarity scores (which are incomparable across systems), RRF uses only the **rank position** of each document in each result list.

For a document at rank `r` in a result list:

$$\text{RRF score} = \sum_{\text{retriever}} \frac{1}{k + r}$$

Where `k=60` is a smoothing constant. Documents that appear near the top in multiple retrieval systems receive the highest combined score. RRF is robust, parameter-light, and consistently outperforms score-based fusion in benchmarks.


In [ ]:
def reciprocal_rank_fusion(result_lists: list[list[Document]],
                           k: int = 60) -> list[Document]:
    """
    Combine multiple ranked result lists using Reciprocal Rank Fusion.

    Each document's RRF score is the sum of 1/(k + rank) across all
    result lists it appears in. Documents that rank well in multiple
    retrievers receive the highest scores.

    Args:
        result_lists: List of ranked document lists (one per retriever).
        k:            Smoothing constant (default 60, per the RRF paper).

    Returns:
        List of Documents sorted by descending RRF score (deduplicated).
    """

    # Map document content to its cumulative RRF score
    rrf_scores: dict[str, float] = {}

    # Map content hash to the Document object for reconstruction
    content_to_doc: dict[str, Document] = {}

    for result_list in result_lists:
        for rank, doc in enumerate(result_list, start=1):
            # Use page_content as the deduplication key
            doc_id = doc.page_content

            # Accumulate RRF score: 1 / (k + rank)
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k + rank))

            # Store reference to Document object
            content_to_doc[doc_id] = doc

    # Sort documents by descending RRF score
    sorted_doc_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)

    return [content_to_doc[doc_id] for doc_id in sorted_doc_ids]


def hybrid_retrieve(query: str,
                    vectorstore: FAISS,
                    bm25_index: BM25Okapi,
                    chunks: list[Document],
                    top_k: int = 10) -> list[Document]:
    """
    Perform hybrid retrieval: FAISS vector search + BM25 keyword search,
    combined via Reciprocal Rank Fusion.

    Args:
        query:       The (rewritten) user query.
        vectorstore: The FAISS vector store.
        bm25_index:  The BM25 index.
        chunks:      The original document chunks.
        top_k:       Number of candidates to retrieve from each retriever.

    Returns:
        RRF-fused list of Documents.
    """

    # Vector similarity retrieval — returns top_k by cosine similarity
    vector_results = vectorstore.similarity_search(query, k=top_k)

    # BM25 keyword retrieval — returns top_k by BM25 score
    bm25_results = bm25_retrieve(query, bm25_index, chunks, top_k=top_k)

    # Fuse both result lists via Reciprocal Rank Fusion
    fused_results = reciprocal_rank_fusion(
        [vector_results, bm25_results]
    )

    # Return top_k fused results
    return fused_results[:top_k]


print("Hybrid Retrieval (FAISS + BM25 + RRF) functions defined.")

Hybrid Retrieval (FAISS + BM25 + RRF) functions defined.


## 12. Cross-Encoder Re-Ranking

### What is Re-Ranking?

Re-ranking is a **post-retrieval** step that applies a more powerful (but slower) relevance model to the initial retrieval results.

The retrieval phase (FAISS + BM25) is designed for speed — it scans thousands of documents in milliseconds. But speed comes at the cost of precision. The initial top-10 may include chunks that are superficially similar to the query but do not actually answer it.

A **Cross-Encoder** addresses this by:
1. Taking each `(query, document)` pair together as a single input
2. Running a full attention-based model over the combined pair
3. Producing a precise relevance score for each pair

This is fundamentally more powerful than bi-encoder (embedding) similarity, because the model can attend to interactions between query and document tokens.

### Why Re-Rank After Retrieval?

Running a Cross-Encoder over all documents would be impractically slow. By first retrieving the top 10 candidates cheaply, then re-ranking only those 10 with the Cross-Encoder, we get the best of both worlds: fast recall + precise relevance scoring.

### Why This Improves Answer Quality

The final 4 documents passed to the LLM are selected by a model that deeply understands their relevance to the query. This reduces:
- **Context noise** — irrelevant documents that distract the LLM
- **Hallucination** — the LLM is less likely to fabricate when given tightly relevant context
- **Context length pressure** — 4 high-quality documents fit comfortably in the prompt

We use `cross-encoder/ms-marco-MiniLM-L-6-v2`, which was trained on the MS MARCO passage ranking dataset and is a well-established choice for document re-ranking.


In [ ]:
# Load the Cross-Encoder model for re-ranking
# ms-marco-MiniLM-L-6-v2 is trained on passage retrieval and balances
# quality and speed well for this task
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Cross-Encoder Loaded: cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank_documents(query: str,
                     documents: list[Document],
                     top_n: int = 4) -> list[Document]:
    """
    Re-rank retrieved documents using the Cross-Encoder model.

    The Cross-Encoder scores each (query, document) pair jointly,
    giving a precise relevance score. We select the top_n highest
    scoring documents to pass to the LLM.

    Args:
        query:     The (rewritten) query string.
        documents: Candidate documents from hybrid retrieval (top 10).
        top_n:     Number of final documents to return (default 4).

    Returns:
        Top-n Documents re-ranked by Cross-Encoder relevance score.
    """

    if not documents:
        return []

    # Build (query, passage) pairs for the Cross-Encoder
    pairs = [(query, doc.page_content) for doc in documents]

    # Score all pairs — Cross-Encoder processes them as a batch
    scores = cross_encoder.predict(pairs)

    # Sort documents by descending relevance score
    scored_docs = sorted(
        zip(scores, documents),
        key=lambda x: x[0],
        reverse=True
    )

    # Return the top_n most relevant documents
    return [doc for _, doc in scored_docs[:top_n]]


print("Re-ranking function defined.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-Encoder Loaded: cross-encoder/ms-marco-MiniLM-L-6-v2
Re-ranking function defined.


## 13. Prompt Template & Generation Chain

The `PromptTemplate` and generation chain are **preserved from the original pipeline** as instructed.

The only change is that the `context` slot is now filled with **reranked documents** (top 4) instead of raw retrieval results (top 5). This gives the LLM higher-quality, more relevant context for generation.


In [ ]:
# Preserved from the original pipeline — no changes to prompt or chain
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are an Enterprise Knowledge Assistant.

Answer ONLY using the provided context.

If the answer is not available in the context, say that the information is not available.

Context:
{context}

Question:
{question}

Answer:"""
)

# Build the generation chain using LangChain Expression Language (LCEL)
chain = prompt_template | llm

print("Prompt Template and Generation Chain Created")

Prompt Template and Generation Chain Created


## 14. Full Advanced RAG Pipeline

The `ask()` function now orchestrates the complete Advanced RAG pipeline:

```
User Query
    ↓
LLM Query Rewriter
    ↓
Rewritten Query
    ↓
┌──────────────────────────────┐
│  FAISS Vector Search  BM25   │
└───────────────┬──────────────┘
                ↓
   Reciprocal Rank Fusion
                ↓
       Top 10 Documents
                ↓
   Cross-Encoder Re-ranker
                ↓
        Top 4 Documents
                ↓
       Prompt Template
                ↓
         LLM Response
```


In [ ]:
def display_retrieved_docs(docs, title="Retrieved Documents", preview_chars=200):
    """
    Pretty-print retrieved documents in a human-readable format,
    showing source, section heading, and a content preview.
    """
    print(f"\n{title} ({len(docs)} documents)")
    print("=" * 70)

    for i, doc in enumerate(docs, 1):
        source  = doc.metadata.get("source", "Unknown")
        heading = (
            doc.metadata.get("heading_1")
            or doc.metadata.get("heading_2")
            or doc.metadata.get("heading_3")
            or "No heading"
        )
        doc_type = doc.metadata.get("type", "N/A")

        # Clean up the content preview — strip markdown artifacts and image placeholders
        content = doc.page_content.replace("<!-- image -->", "").strip()
        content = " ".join(content.split())  # collapse whitespace/newlines
        preview = content[:preview_chars]
        if len(content) > preview_chars:
            preview += "..."

        print(f"\n[{i}] {source} ({doc_type})")
        print(f"    Section: {heading}")
        print(f"    Preview: {preview}")

    print("\n" + "=" * 70)


def ask(question: str,
        hybrid_top_k: int = 10,
        rerank_top_n: int = 4,
        verbose: bool = False):
    """
    Advanced RAG pipeline: Query Rewriting → Hybrid Retrieval → Re-ranking → Generation.

    Args:
        question:      The user's natural language question.
        hybrid_top_k:  Number of candidates from each retriever before RRF (default 10).
        rerank_top_n:  Number of final documents after re-ranking (default 4).
        verbose:       If True, print intermediate steps for inspection.

    Returns:
        Tuple of (answer string, reranked document list).
    """

    # ── Step 1: Query Rewriting ──────────────────────────────────────────
    # Rewrite the user's query to improve retrieval quality
    rewritten_query = rewrite_query(question)

    if verbose:
        print(f"[Query Rewriting]")
        print(f"  Original:  {question}")
        print(f"  Rewritten: {rewritten_query}\n")

    # ── Step 2: Hybrid Retrieval (FAISS + BM25 + RRF) ───────────────────
    # Retrieve top-k candidates by fusing vector and keyword search
    hybrid_docs = hybrid_retrieve(
        query=rewritten_query,
        vectorstore=vectorstore,
        bm25_index=bm25_index,
        chunks=chunks,
        top_k=hybrid_top_k
    )

    if verbose:
        display_retrieved_docs(hybrid_docs, title="Hybrid Retrieval — Top 10 Candidates")

    # ── Step 3: Cross-Encoder Re-ranking ────────────────────────────────
    # Re-rank the top-10 hybrid results and select the best 4
    reranked_docs = rerank_documents(
        query=rewritten_query,
        documents=hybrid_docs,
        top_n=rerank_top_n
    )

    if verbose:
        display_retrieved_docs(reranked_docs, title="Re-ranked — Final Top 4")

    # ── Step 4: Response Generation ─────────────────────────────────────
    # Build context from reranked documents and invoke the generation chain
    context = "\n\n".join(
        doc.page_content
        for doc in reranked_docs
    )

    # Invoke the preserved prompt template + LLM chain
    response = chain.invoke(
        {
            "context": context,
            "question": question  # pass original question to the LLM
        }
    )

    return response.content, reranked_docs


print("Advanced RAG pipeline function defined.")

Advanced RAG pipeline function defined.


## 15. Pipeline Tests

Run three sample queries through the full Advanced RAG pipeline, printing intermediate steps to verify each stage is functioning correctly.


In [ ]:
# Test Query 1
answer, docs = ask(
    "What is Retrieval Augmented Generation?",
    verbose=True
)

print("=" * 60)
print("ANSWER:")
print(answer)

[Query Rewriting]
  Original:  What is Retrieval Augmented Generation?
  Rewritten: What is Retrieval Augmented Generation in the context of LangChain, specifically how does it combine retrieval and generation capabilities to produce text outputs?


Hybrid Retrieval — Top 10 Candidates (10 documents)

[1] rag_paper.pdf (WHITEPAPER)
    Section: Concepts
    Preview: ## Concepts We will cover the following concepts: - Indexing : a pipeline for ingesting data from a source and indexing it. This usually happens in a separate process. - Retrieval and generation : the...

[2] rag_paper.pdf (WHITEPAPER)
    Section: 2. Retrieval and generation
    Preview: ## 2. Retrieval and generation RAG applications commonly work as follows: 1. Retrieve : Given a user input, relevant splits are retrieved from storage using a . Retriever 2. Generate : A produces an a...

[3] LangGraph_overview.pdf (DOCUMENTATION)
    Section: LangGraph overview
    Preview: LangGraph is focused on the underlying capabilit

In [ ]:
# Test Query 2
answer, docs = ask(
    "How do retrievers work in LangChain?",
    verbose=True
)

print("=" * 60)
print("ANSWER:")
print(answer)

[Query Rewriting]
  Original:  How do retrievers work in LangChain?
  Rewritten: What is the architecture and functionality of retriever components in LangChain, including their role in indexing, querying, and retrieving relevant data from external knowledge sources?


Hybrid Retrieval — Top 10 Candidates (10 documents)

[1] rag_paper.pdf (WHITEPAPER)
    Section: Concepts
    Preview: ## Concepts We will cover the following concepts: - Indexing : a pipeline for ingesting data from a source and indexing it. This usually happens in a separate process. - Retrieval and generation : the...

[2] LangGraph_overview.pdf (DOCUMENTATION)
    Section: LangGraph overview
    Preview: ## LangGraph overview Trusted by companies shaping the future of agents- including Klarna, Uber, J.P. Morgan, and more- LangGraph is a low-level orchestration framework and runtime for building, manag...

[3] LangGraph_overview.pdf (DOCUMENTATION)
    Section: LangChain
    Preview: ## LangChain Provides integrations

In [ ]:
# Test Query 3
answer, docs = ask(
    "What are vector stores?",
    verbose=True
)

print("=" * 60)
print("ANSWER:")
print(answer)

[Query Rewriting]
  Original:  What are vector stores?
  Rewritten: What are vector databases and their role in LangChain for efficient semantic search and information retrieval?


Hybrid Retrieval — Top 10 Candidates (10 documents)

[1] rag_paper.pdf (WHITEPAPER)
    Section: This section is an abbreviated version of the content in the . semantic search tutorial
    Preview: ## This section is an abbreviated version of the content in the . semantic search tutorial If your data is already indexed and available for search (i.e., you have a function to execute a search), or ...

[2] rag_paper.pdf (WHITEPAPER)
    Section: Storing documents
    Preview: ## Storing documents Now we need to index our 66 text chunks so that we can search over them at runtime. Following the , our approach is to the contents of each document split and insert these embeddi...

[3] rag_paper.pdf (WHITEPAPER)
    Section: [Task decomposition is...](https://docs.langchain.com/)
    Preview: ## [Task decomposition 

# 16. Gradio Interface
The interactive UI is updated to reflect the Advanced RAG pipeline.

It now displays:

The rewritten query so users can see how their question was interpreted
The answer from the LLM
The reranked source documents with metadata

In [ ]:
import gradio as gr


def chatbot(question: str):
    """
    Gradio handler function.
    Runs the full Advanced RAG pipeline and returns
    the answer, rewritten query, and source metadata.
    """

    if not question.strip():
        return "Please enter a question.", "", ""

    # Run query rewriting separately so we can display it in the UI
    rewritten = rewrite_query(question)

    # Run the full pipeline (rewriting will be called again internally,
    # but this keeps the UI logic clean)
    answer, docs = ask(question)

    # Format retrieved source metadata for display
    sources = ""
    for i, doc in enumerate(docs, start=1):
        source   = doc.metadata.get("source",    "Unknown")
        doc_type = doc.metadata.get("type",      "N/A")
        heading  = doc.metadata.get("heading_1", doc.metadata.get("heading_2", "N/A"))
        parser   = doc.metadata.get("parser",    "N/A")

        sources += f"### Source {i}\n"
        sources += f"**Document:** {source}\n\n"
        sources += f"**Type:** {doc_type}\n\n"
        sources += f"**Section:** {heading}\n\n"
        sources += f"**Parser:** {parser}\n\n"
        sources += doc.page_content[:500] + "...\n\n---\n\n"

    return answer, f"**Rewritten Query:** {rewritten}", sources


with gr.Blocks(title="Enterprise Knowledge Assistant — Advanced RAG") as demo:

    gr.Markdown(
        """
        # 📚 Enterprise Knowledge Assistant — Advanced RAG

        Ask questions about the LangChain knowledge base.

        **Pipeline:** Query Rewriting → Hybrid Retrieval (FAISS + BM25 + RRF) → Cross-Encoder Re-ranking → LLM Response

        **Powered by:** LangChain · FAISS · BM25 · Docling · HuggingFace · Cross-Encoder · Groq Llama 3.3 70B
        """
    )

    question = gr.Textbox(
        label="Ask a Question",
        placeholder="Example: What is Retrieval Augmented Generation?"
    )

    ask_button = gr.Button("Ask")

    # Show the rewritten query so users understand the pipeline
    rewritten_display = gr.Markdown(label="Query Rewriting")

    answer = gr.Markdown(label="Answer")

    with gr.Accordion("Re-ranked Source Documents", open=False):
        sources = gr.Markdown()

    ask_button.click(
        fn=chatbot,
        inputs=question,
        outputs=[answer, rewritten_display, sources]
    )

    question.submit(
        fn=chatbot,
        inputs=question,
        outputs=[answer, rewritten_display, sources]
    )


demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f5edf85137e4d8702a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f5edf85137e4d8702a.gradio.live
